In [1]:
import numpy as np
import pandas as pd

from src.preprocessing import BRANDS_MAPPING, FA_BRANDS_MAPPING, BRAND_FA_TO_MODEL_MAPPING
from src.preprocessing import transform_models, TRANSMISSIONS_MAPPING, FUEL_MAPPING
from src.preprocessing import CITY_TO_PROVINCE, LOC_FA_TO_ENG, COLOR_FA_TO_ENG, BODY_STATUS_FA_TO_ENG

In [4]:
def load_data(file_path: str = "../data/raw/bama_backup.csv") -> pd.DataFrame:
    # Read the raw data from the specified file path.
    df = pd.read_csv(file_path, encoding="utf-8")

	# flag org columns
    df.rename(columns={
        'title': 'title_org',
        'subtitle': 'subtitle_org',
        'trim': 'trim_org',
        'year': 'year_org',
        'mileage': 'mileage_org',
        'location': 'location_org',
        'transmission': 'transmission_org',
        'fuel': 'fuel_org',
        'body_color': 'body_color_org',
        'inside_color': 'inside_color_org',
        'body_status': 'body_status_org',
        'body_type': 'body_type_org',
        'cylinder_fa': 'cylinder_fa_org',
        'image_count': 'image_count_org',
        'modified_date': 'modified_date_org',
        'brand': 'brand_org',
        'price': 'price_org',
        'link': 'link_org',
        'scraped_at': 'scraped_at_org'
	},
    inplace=True
	)

	# add listing id
    df.reset_index(inplace=True)
    df.rename(columns={'index': 'listing_id'}, inplace=True)

    # fix brand names
    df['brand_clean'] = df.loc[:, 'brand_org'].str.strip().str.lower().replace(BRANDS_MAPPING).str.upper()

	# extract the model name
    df_title = df['title_org'].str.split('،', expand=True).rename(
    columns={0: 'brand_org', 1: 'model_org'}
	)
    df_title['brand'] = df_title['brand_org'].replace(FA_BRANDS_MAPPING)
    df_title['model_p1'] = df_title['brand_org'].replace(BRAND_FA_TO_MODEL_MAPPING)

    df_model = transform_models(df_title)

	# fix remaining model names manually
    df_model.loc[4178, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[6058, 'model_org_clean'] = "Saipagroup Shahin Manual G"
    df_model.loc[13654, 'model_org_clean'] = "Irankhodro Samand LX"
    df_model.loc[14376, 'model_org_clean'] = "Saipagroup Pride SD"
    df_model.loc[14875, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[15942, 'model_org_clean'] = "Saipagroup Shahin Manual G"
    df_model.loc[16513, 'model_org_clean'] = "Irankhodro Dena Plus"
    df_model.loc[16879, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[22266, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[27774, 'model_org_clean'] = "Saipagroup Shahin Manual G"
    df_model.loc[29193, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[30790, 'model_org_clean'] = "Hafei Lobo"
    df_model.loc[32865, 'model_org_clean'] = "Dongfeng H30 Cross"
    df_model.loc[35247, 'model_org_clean'] = "Saipagroup Atlas Manual"
    df_model.loc[35266, 'model_org_clean'] = "Irankhodro Pride"

	# add model name
    df['model_clean'] = df_model['model_p1'] + " " + df_model['model_org_clean']
    df['model_clean'] = df['model_clean'].str.strip()
    
	# fix dates data types
    df['modified_date_clean'] = pd.to_datetime(df.loc[:, 'modified_date_org'].str.slice(0, 10))
    df['scraped_at_clean'] = pd.to_datetime(df.loc[:, 'scraped_at_org'])

	# fix mileage data
    df['kilometer_e-03_clean'] = (df['mileage_org'] / 1e03).astype(float) # make it scientific notation
    df['is_zero_kilometer'] = df['kilometer_e-03_clean'] == 0

	# fix transmission data
    df['transmission_clean'] = df['transmission_org'].map(TRANSMISSIONS_MAPPING).str.strip()

	# fix fuel data
    df['fuel_clean'] = df['fuel_org'].map(FUEL_MAPPING).str.strip()

	# fix location data
    df['city_clean'] = df['location_org'].str.split(
    '/', expand=True
	)[0].map(LOC_FA_TO_ENG).str.strip()
    df['province_clean'] = df['city_clean'].map(CITY_TO_PROVINCE).str.strip()

	# fix color data
    df['body_color_clean'] = df['body_color_org'].map(COLOR_FA_TO_ENG).str.strip()
    df['inside_color_clean'] = df['inside_color_org'].map(COLOR_FA_TO_ENG).str.strip()

	# fix body status data
    df['body_status_clean'] = df['body_status_org'].map(BODY_STATUS_FA_TO_ENG).str.strip()

	# fix body type data
    df['body_type_clean'] = df['body_type_org'].str.strip().str.title()

	# handle missing values
    # fuel column
    df.loc[35618, 'fuel_org'] = 'بنزینی'
    df.loc[38383, 'fuel_org'] = 'بنزینی'
    df.loc[38403, 'fuel_org'] = 'بنزینی'
    df.loc[38998, 'fuel_org'] = 'بنزینی'

	# cylinder_fa column
    df_missing = df.loc[df['cylinder_fa_org'].isnull()]
    df_missing.loc[
		df_missing['fuel_org'] == 'برقی', 'cylinder_fa_org'
	] = "0"

	# df_missing.loc[df_missing['fuel'] == 'بنزینی', 'title']
    
    return df

df = load_data()
df

,listing_id,title_org,subtitle_org,trim_org,year_org,mileage_org,location_org,transmission_org,fuel_org,body_color_org,...,kilometer_e-03_clean,is_zero_kilometer,transmission_clean,fuel_clean,city_clean,province_clean,body_color_clean,inside_color_clean,body_status_clean,body_type_clean
0,0,سیتروئن، C3 XR,1404 | تیپ V1,تیپ V1,1404,0,تهران / ستارخان,اتوماتیک,بنزینی,مشکی,...,0.0,True,Automatic,ICE,Tehran,Tehran,Black,Black,Original Paint,Crossover
1,1,لوکانو، L7,1405 | 1.6 لیتر توربو,1.6 لیتر توربو,1405,0,تهران / میرداماد,اتوماتیک,بنزینی,سفید,...,0.0,True,Automatic,ICE,Tehran,Tehran,White,Black,Original Paint,Crossover
2,2,هیوندای، النترا,2025 | 1.5 لیتر,1.5 لیتر,2025,0,تهران / ستارخان,اتوماتیک,بنزینی,سفید,...,0.0,True,Automatic,ICE,Tehran,Tehran,White,Black,Original Paint,Passenger_Car
3,3,چانگان، CS 55 پلاس,2024 | 1.5 لیتر توربو,1.5 لیتر توربو,2024,28000,تهران / ولنجک,اتوماتیک,بنزینی,خاکستری,...,28.0,False,Automatic,ICE,Tehran,Tehran,Gray,Black,Original Paint,Crossover
4,4,کی ام سی، T8,1403,دنده ای,1403,30000,تهران / میرداماد,دنده ای,بنزینی,مشکی,...,30.0,False,Manual,ICE,Tehran,Tehran,Black,Black,Original Paint,Pickup
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39656,39656,ب ام و، سری 5 سدان,2015 | 528i,528i,2015,77000,اصفهان / سعادت آباد,اتوماتیک,بنزینی,سفید,...,77.0,False,Automatic,ICE,Isfahan,Isfahan,White,Maroon,Original Paint,Passenger_Car
39657,39657,ب ام و، سری 5 سدان,2018 | 530i,530i,2018,49000,اصفهان / خیابان نظر غربی,اتوماتیک,بنزینی,مشکی,...,49.0,False,Automatic,ICE,Isfahan,Isfahan,Black,Mocha,Original Paint,Passenger_Car
39658,39658,نیسان، ترا,1404 | اکسکلوسیو,اکسکلوسیو,1404,0,اصفهان / سعادت آباد,اتوماتیک,بنزینی,مشکی,...,0.0,True,Automatic,ICE,Isfahan,Isfahan,Black,Maroon,Original Paint,Suv
39659,39659,ب ام و، سری 7,2017 | 730li,730li,2017,42000,اصفهان / خیابان نظر غربی,اتوماتیک,بنزینی,سفید صدفی,...,42.0,False,Automatic,ICE,Isfahan,Isfahan,Pearl White,Maroon,Original Paint,Passenger_Car


In [3]:
df.columns

Index(['listing_id', 'title_org', 'subtitle_org', 'trim_org', 'year_org',
       'mileage_org', 'location_org', 'transmission_org', 'fuel_org',
       'body_color_org', 'inside_color_org', 'body_status_org',
       'body_type_org', 'cylinder_fa_org', 'image_count_org',
       'modified_date_org', 'brand_org', 'price_org', 'link_org',
       'scraped_at_org', 'brand_clean', 'model_clean', 'modified_date_clean',
       'scraped_at_clean', 'kilometer_e-03_clean', 'is_zero_kilometer',
       'transmission_clean', 'fuel_clean', 'city_clean', 'province_clean',
       'body_color_clean', 'inside_color_clean', 'body_status_clean'],
      dtype='str')